<a href="https://colab.research.google.com/github/EngMohamed-op/supervised-and-unsupervised-project/blob/main/WorldCup2026Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#World Cup 2026 — Match Prediction & Tournament Simulator
##UNIT 4&5 Machine Learning Project — Football Analytics



**Dataset 1** : International football results from 1872 to 2026

**Source** : Kaggle - [International football results from 1872 to 2026](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)
____________________________________________

**Dataset 2** : FIFA World Ranking 1992-2024

**Source** : Kaggle - [FIFA World Ranking 1992-2024](https://www.kaggle.com/datasets/cashncarry/fifaworldranking)

#Project outline :
1. Dataset description & problem definition
2. Data loading & structure
3. Data cleaning & preprocessing
4. Elo engine & feature engineering
5. Exploratory data analysis (EDA)
6. Machine learning models
7. World Cup simulation (Monte Carlo)
8. Deployment — FastAPI + Streamlit App

#Step 1 — Dataset Description & Problem Definition :

This project uses 5 interconnected football datasets covering every international match since 1872, FIFA ranking snapshots from 1992 to 2024, individual goal events, penalty shoot-out records, and historical country name changes. Together they allow us to reconstruct the full strength history of every national team and simulate the FIFA World Cup 2026 with probabilistic, data-driven match predictions.


1. International Football Results (1872 - 2026)
What is this dataset about?
This dataset contains a complete record of international football matches starting from the first official match in 1872 up to 2026. It includes essential details such as home and away teams, scores, tournament types (from friendlies to FIFA World Cups), match dates, and the specific city and country where each game was played. It provides a historical timeline of how the sport has grown and shifted globally over more than 150 years.

kaggle [International football results from 1872 to 2026](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)



2. FIFA World Ranking (1992 - 2024)
What is this dataset about?
This dataset tracks the official FIFA World Rankings for national teams from their inception in 1992 through 2024. It includes key metrics such as the team’s total points, current rank, previous rank, and the federation they belong to (e.g., UEFA, AFC). This data allows for the analysis of team consistency, the rise of emerging nations, and how FIFA’s points system has evaluated the strength of national teams over the last three decades.

kaggle : [FIFA World Ranking 1992-2024](https://www.kaggle.com/datasets/cashncarry/fifaworldranking)

#Main questions we will explore


*   Can Elo ratings reliably predict the winner of a football match between any two national teams?
*   Which team is most likely to win the 2026 World Cup, and with what probability?
*   Do stronger teams (higher Elo) consistently beat weaker ones, or do upsets follow a pattern?
*   How does home advantage quantifiably affect match outcomes?
*   Which confederation produces the most consistently elite teams over 20+ years?
*   Can we cluster national teams into meaningful tiers (strong / medium / weak) using unsupervised learning?



#What insights do we expect?


*   Elo difference will be the single strongest predictor of match outcome, outperforming raw FIFA rank.
*   Argentina & France will emerge as co-favourites in the Monte Carlo simulation with 1,000+ runs.
*   Home advantage adds roughly +100 Elo points of effective strength — measurable and statistically significant.
*   UEFA & CONMEBOL teams will cluster clearly above other confederations in the unsupervised tier model.


##Step 2A — Install & Import Libraries

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import pearsonr
from collections import defaultdict
from google.colab import files


#Step 2B — Load the Dataset

In [3]:
uploaded = files.upload()

In [8]:
# SECTION 1- DATA LOADING & PREPROCESSING
print("_" * 60)
print(" 1- Loading datasets : ")
print("_" * 60)

df_results    = pd.read_csv(f"results.csv")
df_fifa       = pd.read_csv(f"fifa_ranking-2024-06-20.csv")
df_goals      = pd.read_csv(f"goalscorers.csv")
df_shootouts  = pd.read_csv(f"shootouts.csv")
df_former     = pd.read_csv(f"former_names.csv")

print("\nTable Shapes:")
for name, df in [("results", df_results), ("fifa_ranking", df_fifa),
                 ("goalscorers", df_goals), ("shootouts", df_shootouts),
                 ("former_names", df_former)]:
    print(f"   {name:20s}  {df.shape[0]:>6,} rows × {df.shape[1]} cols")


____________________________________________________________
 1- Loading datasets : 
____________________________________________________________


FileNotFoundError: [Errno 2] No such file or directory: 'goalscorers.csv'

#Step 3 — Data Structure and Data Cleaning (per table)

#1. Results table (results.csv) : df_result

In [ ]:
df_results.head()

In [ ]:
df_results.info()

In [ ]:
missing = df_results.isnull().sum()
missing_pct = (missing / len(df_results) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('═' * 45)
print('  df_results - MISSING VALUES REPORT')
print('═' * 45)
print(missing_df.to_string())
print(f'\n Total columns with missing data: {len(missing_df)}')

In [ ]:
# ──────────────────────────────────────────
# 2-A  results.csv :

print("\n" + "=" * 60)
print("  SECTION 2 — DATA CLEANING (Result)")
print("=" * 60)



# date is stored as a string > Convert datas to datatime object
df_results['date'] = pd.to_datetime(df_results['date'])


# Clean string data (remove leading/trailing whitespaces cause silent merge failures
df_results['home_team'] = df_results['home_team'].str.strip()
df_results['away_team'] = df_results['away_team'].str.strip()
df_results['tournament'] = df_results['tournament'].str.strip()


# exact duplicates (same match logged twice) skew the Elo engine
before = len(df_results)
df_results.drop_duplicates(subset=['date','home_team','away_team'], inplace=True)
print(f"\n[results] Removed {before - len(df_results):,} exact-duplicate rows")

# negative scores are impossible; flag them
neg_scores = df_results[(df_results['home_score'] < 0) | (df_results['away_score'] < 0)]
print(f"[results] Negative score rows found: {len(neg_scores)}")


# Filter data from 2002 onwards and sort chronologically
df_results = df_results[df_results['date'] >= '2002-01-01'].sort_values('date').reset_index(drop=True)
print(f"[results] Rows after 2002 filter: {len(df_results):,}")


# 2. TEAM NAME MAPPING
# the same country appears under different names across the two Ensures consistency between match results and FIFA rankings

name_mapping = {
    'USA':            'United States',
    'Korea Republic': 'South Korea',
    'IR Iran':        'Iran',
    'Korea DPR':      'North Korea',
    'China PR':       'China',
    'Czechia':        'Czech Republic',
    'Türkiye':        'Turkey',
    'Ivory Coast':    "Côte d'Ivoire",
    'DR Congo':       'Congo DR',
    'Cape Verde':     'Cabo Verde',
    'Kyrgyzstan':     'Kyrgyz Republic',
    'Vietnam':        'Viet Nam',
}
df_results['home_team'] = df_results['home_team'].replace(name_mapping)
df_results['away_team'] = df_results['away_team'].replace(name_mapping)


print(f"[results] {len(name_mapping)} team name inconsistencies harmonised")


#2. fifa_ranking-2024-06-20 table (fifa_ranking-2024-06-20.csv) : df_fifa

In [9]:
df_fifa.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,140.0,Brunei Darussalam,BRU,2.0,0.0,140,AFC,1992-12-31
1,33.0,Portugal,POR,38.0,0.0,33,UEFA,1992-12-31
2,32.0,Zambia,ZAM,38.0,0.0,32,CAF,1992-12-31
3,31.0,Greece,GRE,38.0,0.0,31,UEFA,1992-12-31
4,30.0,Algeria,ALG,39.0,0.0,30,CAF,1992-12-31


In [ ]:
df_fifa.info()

In [ ]:
df_fifa.isnull().sum()

In [ ]:
missing = df_fifa.isnull().sum()
missing_pct = (missing / len(df_fifa) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('_' * 45)
print('  df_fifa - MISSING VALUES REPORT')
print('_' * 45)
print(missing_df.to_string())
print(f'\n Total columns with missing data: {len(missing_df)}')

In [ ]:
# ──────────────────────────────────────────
# 2-B  fifa_ranking.csv


# date is stored as a string > Convert datas to datatime object
df_fifa['rank_date']     = pd.to_datetime(df_fifa['rank_date'])

# Clean string data (remove leading/trailing whitespaces cause silent merge failures
df_fifa['country_full']  = df_fifa['country_full'].str.strip()

#handle missing value :

# 9 rows have NaN rank , these are incomplete records; drop them
before = len(df_fifa)
df_fifa.dropna(subset=['rank'], inplace=True)
print(f"\n[fifa_ranking] Dropped {before - len(df_fifa)} rows with null rank")

# total_points is the only column we use for Elo seeding;
# fill the extremely rare zero-point edge case with the confederate median
df_fifa['total_points'] = df_fifa['total_points'].fillna(
    df_fifa.groupby('confederation')['total_points'].transform('median')
)

df_fifa.drop_duplicates(subset=['rank_date','country_full'], inplace=True)


print(f"[fifa_ranking] Shape after cleaning: {df_fifa.shape}")



#3. Goalscores table (goalscorers.csv) : df_goals

In [ ]:
df_goals.head()

In [ ]:
df_goals.info()

In [ ]:
missing = df_goals.isnull().sum()
missing_pct = (missing / len(df_goals) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('_' * 45)
print('  df_goals - MISSING VALUES REPORT')
print('_' * 45)
print(missing_df.to_string())
print(f'\n Total columns with missing data: {len(missing_df)}')

In [ ]:
# ──────────────────────────────────────────
# 2-C  goalscorers.csv


# date is stored as a string > Convert datas to datatime object
df_goals['date'] = pd.to_datetime(df_goals['date'])

# Clean string data (remove leading/trailing whitespaces cause silent merge failures
df_goals['home_team'] = df_goals['home_team'].str.strip()
df_goals['away_team'] = df_goals['away_team'].str.strip()
df_goals['team']      = df_goals['team'].str.strip()
df_goals['scorer']    = df_goals['scorer'].str.strip()

#48 rows have no scorer name , these are data-entry gaps.
#We keep them for aggregate stats (goals per match) but flag them.
missing_scorers = df_goals['scorer'].isna().sum()
print(f"\n[goalscorers] Missing scorer names: {missing_scorers}")

#minute=NaN means event time is unknown — impute with median
#so that any minute-based analysis is not skewed.
median_min = df_goals['minute'].median()
df_goals['minute'] = df_goals['minute'].fillna(median_min)
print(f"[goalscorers] Imputed {256} null minutes with median={median_min:.0f}'")

#filter to same 2002+ window to stay consistent with results
df_goals = df_goals[df_goals['date'] >= '2002-01-01'].reset_index(drop=True)


print(f"[goalscorers] Rows after 2002 filter: {len(df_goals):,}")


#4. Shootouts table (shootouts.csv) : df_shootouts

In [ ]:
df_shootouts.head()

In [ ]:
df_shootouts.info()

In [ ]:
missing = df_shootouts.isnull().sum()
missing_pct = (missing / len(df_shootouts) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('_' * 45)
print('  df_shootouts - MISSING VALUES REPORT')
print('_' * 45)
print(missing_df.to_string())
print(f'\n Total columns with missing data: {len(missing_df)}')

In [ ]:
# ──────────────────────────────────────────
# 2-D  shootouts.csv

# date is stored as a string > Convert datas to datatime object
df_shootouts['date'] = pd.to_datetime(df_shootouts['date'])

# Clean string data (remove leading/trailing whitespaces cause silent merge failures
df_shootouts['home_team'] = df_shootouts['home_team'].str.strip()
df_shootouts['away_team'] = df_shootouts['away_team'].str.strip()
df_shootouts['winner']    = df_shootouts['winner'].str.strip()

#425/665 rows have no first_shooter — we won't use this column
# for modeling, so we just note the gap.
print(f"\n[shootouts] Missing first_shooter: {df_shootouts['first_shooter'].isna().sum()}")
df_shootouts = df_shootouts[df_shootouts['date'] >= '2002-01-01'].reset_index(drop=True)


print(f"[shootouts] Rows after 2002 filter: {len(df_shootouts):,}")


#5. former_names table (former_names.csv) : df_former

In [ ]:
df_former.head()

In [ ]:
df_former.info()

In [ ]:
missing = df_former.isnull().sum()
missing_pct = (missing / len(df_former) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print('_' * 45)
print('  df_former - MISSING VALUES REPORT')
print('_' * 45)
print(missing_df.to_string())
print(f'\n Total columns with missing data: {len(missing_df)}')

In [ ]:
# ──────────────────────────────────────────
# 2-E  former_names.csv

#used only for historical reference; no cleaning needed beyond type conversion.
df_former['start_date'] = pd.to_datetime(df_former['start_date'])
df_former['end_date']   = pd.to_datetime(df_former['end_date'])
print(f"\n[former_names] {len(df_former)} historical name changes loaded — no nulls.")



All tables cleaned successfully !

#Ster 4 — ELO ENGINE (Helper Functions) :

In [ ]:
# SECTION 3 — ELO ENGINE (HELPER FUNCTIONS)
#_______________________________________________________

print("=" * 60)
print("  SECTION 3 — ELO ENGINE")
print("=" * 60)

# Function to fetch points for "New Teams" at their first historical appearance
def fetch_initial_points(team_name, debut_date):
    """
    Some national teams appear in results.csv for the first time
         after 2002, so they have no Elo seed. We query their earliest
         FIFA ranking record on or after their debut date. If none
         exists we fall back to 1400 (slightly below world average).
    """
    record = df_fifa[(df_fifa['country_full'] == team_name) &
                     (df_fifa['rank_date'] >= debut_date)].sort_values('rank_date')
    return record.iloc[0]['total_points'] if not record.empty else 1400

# K-factor : Function to determine Tournament Importance
def get_k_factor(tournament):
    """
         Not every match carries the same weight.
         A World Cup final has far more predictive value than a friendly.
         K-factor scales how much a result moves the Elo needle.
    """
    if 'FIFA World Cup' in tournament and 'Qualification' not in tournament:
        return 60   # Maximum weight — elite competition
    if any(t in tournament for t in ['Euro', 'Copa América', 'Asian Cup',
                                      'Africa Cup', 'Gold Cup', 'Nations League']):
        return 40   # Continental championships
    if 'Qualification' in tournament:
        return 30   # Competitive but not a finals tournament
    return 20       # Friendlies / minor tournaments

# Goal-margin factor (G) :
def get_goal_margin_factor(h_s, a_s):
    """
        Winning 5-0 is stronger evidence than winning 1-0.
         The G-factor boosts/dampens Elo change proportionally.
         Formula mirrors the World Football Elo Rating standard.
    """
    diff = abs(h_s - a_s)
    if diff <= 1: return 1.0
    if diff == 2: return 1.5
    return (11 + diff) / 8   # e.g. diff=3 → 1.75, diff=5 → 2.0

# Seed from real 2002 FIFA points
print("\nSeeding Elo from 2002 FIFA rankings...")
fifa_2002  = (df_fifa[df_fifa['rank_date'].dt.year == 2002]
              .sort_values('rank_date')
              .groupby('country_full')
              .first())
elo_ratings = fifa_2002['total_points'].to_dict()
print(f"Teams seeded from 2002: {len(elo_ratings)}")

# Dynamic Elo Engine (Main loop)
print("Running Elo Engine match-by-match (2002 → 2026)…")
home_elo_pre, away_elo_pre = [], []

for idx, row in df_results.iterrows():
    h_team, a_team = row['home_team'], row['away_team']

    # Check if teams are new and fetch their starting points if necessary
    for team in [h_team, a_team]:
        if team not in elo_ratings:
            elo_ratings[team] = fetch_initial_points(team, row['date'])

    # Snapshot ratings BEFORE this match (the feature we train on)
    h_pre = elo_ratings[h_team]
    a_pre = elo_ratings[a_team]
    home_elo_pre.append(h_pre)
    away_elo_pre.append(a_pre)

    # Home advantage: +100 Elo points unless the venue is neutral
    h_elo_eff = h_pre + (100 if not row['neutral'] else 0)

    # Expected win probability for home team (logistic formula)
    exp_h = 1 / (1 + 10 ** ((a_pre - h_elo_eff) / 400))

    # Actual result: 1=win, 0.5=draw, 0=loss
    s_h = (1   if row['home_score'] > row['away_score'] else
           0.5 if row['home_score'] == row['away_score'] else 0)

    # Claculate Elo update
    k      = get_k_factor(row['tournament'])
    g      = get_goal_margin_factor(row['home_score'], row['away_score'])
    update = k * g * (s_h - exp_h)

    # Update Ratings in the dictionary
    elo_ratings[h_team] += update
    elo_ratings[a_team] -= update

# Assemble final training DataFrame @
# Feature Engineering
print("Assembling final_training_data_2026.csv …")
df_results['home_elo_pre'] = home_elo_pre
df_results['away_elo_pre'] = away_elo_pre
df_results['elo_diff']     = df_results['home_elo_pre'] - df_results['away_elo_pre']

# Target variable , what the model will predict
# 3-class label (Home Win / Draw / Away Win) is the simplest formulation that covers all possible match outcomes.
def encode_result(row):
    if   row['home_score'] > row['away_score']: return 2   # Home Win
    elif row['home_score'] == row['away_score']: return 1  # Draw
    else:                                        return 0   # Away Win

df_results['result'] = df_results.apply(encode_result, axis=1)
df_results['result_label'] = df_results['result'].map({2:'Home Win',1:'Draw',0:'Away Win'})

# Export the final dataset for Machine Learning
output_file = 'final_training_data_2026.csv'
df_results.to_csv(output_file, index=False)

print(f"Saved  final_training_data_2026.csv  ({len(df_results):,} rows)\n")

# Save final Elo snapshot (used by the simulator in Part 2)
elo_snapshot = pd.DataFrame(
    list(elo_ratings.items()), columns=['team', 'elo_2026']
).sort_values('elo_2026', ascending=False).reset_index(drop=True)
elo_snapshot.to_csv("elo_snapshot_2026.csv", index=False)
print("Saved  elo_snapshot_2026.csv  (used by Part 2 Simulator)")


#Step 5 — Exploratory Data Analysis (EDA)

In [ ]:

# SECTION 4-EXPLORATORY DATA ANALYSIS


print("\n" + "_" * 60)
print("EDA")
print("_" * 60)

df = df_results.copy()   # main working frame for EDA

# Helper: to save figure
def savefig(name):
    path = f"eda_plots/{name}.png"
    plt.savefig(path, bbox_inches='tight', facecolor=plt.rcParams['figure.facecolor'])
    plt.close()
    print(f"    Saved: {path}")
